In [0]:
import os
import sys
from pathlib import Path

# En Serverless, indicar la raíz absoluta del proyecto completo en /Workspace/...
# Puede fijarse aquí o mediante RISKVISION_PROJECT_ROOT; no debe apuntar a notebooks/.
PROJECT_ROOT_OVERRIDE = os.environ.get("RISKVISION_PROJECT_ROOT")
if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser()
    if not project_root.is_absolute():
        raise ValueError("RISKVISION_PROJECT_ROOT debe ser una ruta absoluta")
    project_root = project_root.resolve()
else:
    # Compatibilidad local: iniciar desde la raíz o desde notebooks/.
    candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
    project_root = next((p for p in candidates if (p / "src" / "proyecto").is_dir()), None)
if project_root is None or not (project_root / "src" / "proyecto" / "config.py").is_file():
    raise RuntimeError("No se encuentra src/proyecto: configura RISKVISION_PROJECT_ROOT con la raíz absoluta")
source_dir = str(project_root / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import proyecto
if Path(proyecto.__file__).resolve().parent != project_root / "src" / "proyecto":
    raise RuntimeError("El kernel tiene cargada otra copia de proyecto; reinícialo y revisa la raíz")
print("Raíz verificada:", project_root)
print("Módulos:", Path(proyecto.__file__).resolve().parent)
print("Python:", sys.version.split()[0], "| plataforma:", sys.platform)

# El kernel de Serverless precarga paquetes del runtime (pandas 2.3.3,
# typing_extensions antiguo, etc.) en sys.modules ANTES de activar el entorno
# del proyecto. Eso impide importar las versiones del entorno y el diagnóstico
# abortaría. Purga y recarga: todo módulo precargado desde
# /databricks/python/lib cuya distribución exista también en el entorno del
# proyecto se reimporta desde ahí. No se alteran métodos internos ni se
# instalan paquetes. Efecto por sesión; reiniciar la sesión lo deshace.
import importlib.metadata as _im
from pathlib import Path as _Path
_venv_sp = next((_Path(_p) for _p in sys.path
                 if "envs" in _p and _p.rstrip("/").endswith("site-packages")), None)
if _venv_sp is not None and _venv_sp.is_dir():
    _nombres_venv = {(_q.name[:-3] if _q.name.endswith(".py") else _q.name)
                     for _q in _venv_sp.iterdir()}
    _a_purgar = sorted(_n for _n, _m in list(sys.modules.items())
                       if getattr(_m, "__file__", None)
                       and "/databricks/python/lib/" in _m.__file__
                       and _n.split(".")[0] in _nombres_venv)
    for _n in _a_purgar:
        del sys.modules[_n]
    if _a_purgar:
        print("Módulos del runtime reemplazados por los del entorno del proyecto:",
              len(_a_purgar), "->", sorted({_n.split(".")[0] for _n in _a_purgar}))
        _p_instalado = _im.version("pandas")
        import pandas
        assert pandas.__version__ == _p_instalado, "la recarga no tomó la versión del entorno"
from proyecto.runtime import diagnose_environment
workspace_environment = ("DATABRICKS_RUNTIME_VERSION" in os.environ
                         or project_root.as_posix().startswith(("/Workspace/", "/Volumes/")))
ENVIRONMENT_MODE = os.environ.get("RISKVISION_ENVIRONMENT_MODE",
                                  "serverless" if workspace_environment else "local")
if ENVIRONMENT_MODE not in {"local", "serverless"}:
    raise ValueError("RISKVISION_ENVIRONMENT_MODE debe ser local o serverless")
if workspace_environment and ENVIRONMENT_MODE != "serverless":
    raise ValueError("En Databricks se exige el diagnóstico serverless")
ENVIRONMENT_DIAGNOSTIC = diagnose_environment(
    project_root / "requirements-serverless-06.txt", strict=ENVIRONMENT_MODE == "serverless")
print("Diagnóstico previo a datos:", ENVIRONMENT_DIAGNOSTIC)

from proyecto.inference.service import (InferenceControls, open_session, predict_clients,
    publish_predictions, synthetic_clients)
from proyecto.inference.predict import InputValidationError
from proyecto.models.persistence import validate_input_schema
from proyecto.tracking.reporting import code_revision
import json

# Inferencia real autorizada por la usuaria en esta sesión sobre el bundle
# aprobado definitivamente por la etapa 08; la publicación queda desactivada.
DIAGNOSTIC_MODE = False
ALLOW_REAL_INFERENCE = True
ALLOW_REAL_PREDICTION_PUBLICATION = False
ALLOW_TRAINING = False
ALLOW_TEST_ACCESS = False
RUN_SYNTHETIC_EXAMPLES = False
CONTROLS = InferenceControls(diagnostic_mode=DIAGNOSTIC_MODE,
    allow_real_inference=ALLOW_REAL_INFERENCE,
    allow_real_publication=ALLOW_REAL_PREDICTION_PUBLICATION,
    allow_training=ALLOW_TRAINING, allow_test_access=ALLOW_TEST_ACCESS)


# 09 · Inferencia — del pipeline guardado a un resultado comprensible

RiskVision, Grupo 6. Se conserva la secuencia académica: rutas, recuperación del modelo, esquema, predicción, comprobación y ficha para una futura aplicación.

Entrada de cliente **ya limpia según el contrato del proyecto** → validación → transformaciones y codificación del pipeline → modelo → probabilidad de incumplimiento → clasificación por umbral explícito → resultado con trazabilidad.

El modelo LightGBM original con paquete E continúa **PROVISIONAL**. No hay un modelo definitivo aprobado ni resultados reales completos de 07. No se declara un champion ni se descarga un alias automáticamente. La referencia y sus parámetros no se modifican.

Este notebook no entrena modelos, no lee datasets, no optimiza ni registra modelos. El diagnóstico requiere un paquete sintético creado previamente en un temporal por las pruebas de 08; el notebook solo lo recupera. Los controles están desactivados por defecto y no se generan predicciones reales ni archivos.


## 0 · Rutas y permisos explícitos

Reutilizar Environment y requirements-serverless-06.txt; no se requieren dependencias nuevas. La primera celda configura RISKVISION_PROJECT_ROOT y diagnostica versiones/imports antes de cualquier deserialización. Configurar BUNDLE_DIRECTORY con una carpeta de paquete 08, no con un Joblib aislado. Conservar la huella del manifiesto recibida por un canal confiable; nunca inferir confianza porque el archivo esté en el mismo directorio.

No se carga entrenamiento ni prueba. Para producción hace falta autorización de inferencia y un modelo aprobado de alcance real. El diagnóstico solo admite un artefacto sintético provisional y un umbral declarado sintético; no admite datos reales. El llamador debe garantizar el origen sintético de sus entradas: el programa no puede deducirlo por sus valores.


In [0]:
# Bundle aprobado definitivamente por la etapa 08 (recibo de esta sesión;
# huella externa del manifiesto tal como la entregó 08).
BUNDLE_DIRECTORY = "/Workspace/Users/estefaniacg1995@gmail.com/riskvision/bundles/08-candidate-20261107T002331Z"
EXPECTED_MANIFEST_SHA256 = "9dca9a439a904e13f2317fb9a742aca18b9d6584783a1021ea34efa20afcc8c5"
TRUST_ARTIFACT_SOURCE = True
SYNTHETIC_THRESHOLD = None  # solo diagnóstico: por ejemplo 0.5, sin aprobación económica
REAL_CLIENT_INPUTS = None  # proporcionar únicamente tras aprobación
PREDICTION_OUTPUT = None  # archivo NUEVO; sin exportación por defecto
SESSION = None
SINGLE_RESULT = BATCH_RESULT = MISSING_RESULT = REAL_RESULT = None
INPUT_WARNINGS = []
INVALID_INPUT_MESSAGE = None
REVISION = code_revision(project_root)
print("Revisión:", REVISION, "| controles:", CONTROLS)


## 1 · Recuperar el paquete verificado de etapa 08

Se usa load_bundle mediante open_session: confirma procedencia, verifica hashes de los cuatro archivos antes de Joblib, comprueba versiones, pipeline ajustado completo y concordancia de esquema/parámetros. No hay fallback silencioso a otro modelo ni exportación automática.

En modo real se exige approved_definitive y la aprobación documentada de 08. Se recupera exclusivamente el umbral de la decisión congelada de 07; si no existe, el flujo se detiene. Ningún valor 0.5 o 1/6 se elige como umbral definitivo en esta etapa. Activar un control no aprueba un provisional.


In [0]:
if CONTROLS.diagnostic_mode or CONTROLS.allow_real_inference:
    if BUNDLE_DIRECTORY is None:
        raise ValueError("Falta la carpeta explícita del paquete verificado de etapa 08")
    SESSION = open_session(BUNDLE_DIRECTORY, trusted=TRUST_ARTIFACT_SOURCE,
        manifest_sha256=EXPECTED_MANIFEST_SHA256, controls=CONTROLS,
        synthetic_threshold=SYNTHETIC_THRESHOLD)
    print("Paquete recuperado:", SESSION.bundle.metadata["model_name"],
          SESSION.bundle.metadata["model_version"], SESSION.bundle.metadata["status"])
else:
    print("Carga desactivada. No se leyó ningún artefacto ni dataset.")


## 2 · Esquema de entrada: reutilizar, no reconstruir

El esquema viaja dentro del paquete 08; no se vuelve a construir con entrenamiento ni con clientes nuevos. Se requieren exactamente los 23 predictores originales, sin ID, objetivo ni variables derivadas. Las derivadas E y todo lo aprendido se calculan en el pipeline guardado.

Los códigos de educación y matrimonio deben estar ya agrupados por la limpieza fija del proyecto. Se rechazan texto numérico, booleanos, infinitos, categorías inválidas, edades no enteras, valores imposibles y columnas duplicadas/adicionales. Faltar una columna es un error; un valor faltante dentro de una columna existente utiliza el imputador aprendido. Se permiten facturas negativas plausibles, faltantes y categorías válidas ausentes del ajuste (con advertencia). Salirse del rango observado produce advertencia, no una nueva regla inventada de rechazo. El orden de columnas se restaura; el orden e índice de filas se conservan.


In [0]:
INPUT_SCHEMA = None if SESSION is None else SESSION.bundle.schema
if INPUT_SCHEMA is not None:
    print("Columnas requeridas:", INPUT_SCHEMA["input_columns"])
    print("Contrato:", INPUT_SCHEMA["contract"])
    print("Umbral:", SESSION.threshold, "| procedencia:", SESSION.threshold_source)
else:
    print("Esquema pendiente de un paquete autorizado; no se crea otro esquema")


## 3 · Predicción individual, por lotes y errores claros

predict_clients acepta un diccionario de un cliente o un DataFrame para un lote. Devuelve probabilidad de incumplimiento, clase (probabilidad >= umbral), texto comprensible, umbral/procedencia, modelo/versión/estado y huella. Las advertencias quedan en attrs y la salida preserva el índice.

La probabilidad es una estimación, no certeza, explicación causal ni decisión automática de crédito. El umbral sintético solo verifica el funcionamiento del código. No implica calibración comprobada ni ahorro económico bajo la hipótesis 5:1. No se usa el umbral auxiliar de revisión 0.55 del esquema como criterio operativo.


In [0]:
if RUN_SYNTHETIC_EXAMPLES:
    if SESSION is None or not CONTROLS.diagnostic_mode:
        raise PermissionError("Los ejemplos requieren una sesión de diagnóstico sintética explícita")
    example = synthetic_clients()
    SINGLE_RESULT = predict_clients(example.iloc[0].to_dict(), SESSION)
    BATCH_RESULT = predict_clients(example, SESSION)
    missing = example.iloc[:1].copy()
    missing.loc[:, "LIMIT_BAL"] = float("nan")
    MISSING_RESULT = predict_clients(missing, SESSION)
    impossible = example.iloc[:1].copy()
    impossible.loc[:, "AGE"] = 200.
    try:
        predict_clients(impossible, SESSION)
    except InputValidationError as error:
        INVALID_INPUT_MESSAGE = str(error)
    else:
        raise AssertionError("La entrada imposible debía rechazarse")
    INPUT_WARNINGS = BATCH_RESULT.attrs["warnings"]
    print("SOLO DIAGNÓSTICO SINTÉTICO; umbral no aprobado")
    print(SINGLE_RESULT.to_string())
    print(BATCH_RESULT.to_string())
    print("Advertencias:", INPUT_WARNINGS, "| rechazo esperado:", INVALID_INPUT_MESSAGE)
if CONTROLS.allow_real_inference and REAL_CLIENT_INPUTS is not None:
    REAL_RESULT = predict_clients(REAL_CLIENT_INPUTS, SESSION)
    # No imprimir ni exportar resultados reales automáticamente.
if CONTROLS.allow_real_publication:
    if REAL_RESULT is None or PREDICTION_OUTPUT is None:
        raise PermissionError("Falta resultado autorizado y destino explícito nuevo")
    publish_predictions(REAL_RESULT, SESSION, PREDICTION_OUTPUT)


## 4 · Identidad del artefacto y consistencia

No se reevalúa la prueba ni se selecciona un umbral con nuevos clientes. Las huellas enlazan el pipeline con metadatos y esquema de 08; las probabilidades deben coincidir con predict_proba del mismo pipeline. Las pruebas automatizadas comprueban además un proceso Python nuevo y manipulación del artefacto.

Conservar la huella externa y el recibo de 08. La integridad no demuestra calidad, aprobación ni autenticidad de un origen desconocido. Una prueba sintética de ida y vuelta no sustituye la evaluación real pendiente de 07.


In [0]:
SYNTHETIC_CONSISTENCY = None
if BATCH_RESULT is not None:
    import numpy as np
    expected = SESSION.bundle.predict_proba(synthetic_clients())[:, 1]
    np.testing.assert_allclose(BATCH_RESULT["default_probability"].to_numpy(), expected, rtol=1e-14, atol=1e-15)
    np.testing.assert_array_equal(BATCH_RESULT["classification"], (expected >= SESSION.threshold).astype(int))
    SYNTHETIC_CONSISTENCY = {"probabilities_match_pipeline": True, "threshold_applied": True}
    print("Consistencia sintética:", SYNTHETIC_CONSISTENCY)


## 5 · Ficha y evidencia de inferencia

reports/09_inferencia.md describe el protocolo, contrato, pruebas y pendientes. Se conserva una constancia en memoria, sin escribir modelos/ficha_modelo.md ni inventar rendimiento. Cada ejecución futura deberá documentar revisión, artefacto, política operativa y permisos. No se importan resultados de otra rama como propios.


In [0]:
REPORT_09 = {"stage": "09", "status": "implemented_pending_real_approval" if SESSION is None or SESSION.controls.diagnostic_mode else "authorized_inference",
    "revision": REVISION, "model_status": "provisional" if SESSION is None else SESSION.bundle.metadata["status"],
    "diagnostic_only": CONTROLS.diagnostic_mode, "real_inference_allowed": CONTROLS.allow_real_inference,
    "real_publication_allowed": CONTROLS.allow_real_publication,
    "artifact_sha256": None if SESSION is None else SESSION.bundle.metadata["artifact_sha256"],
    "validation_report_id": None if SESSION is None else SESSION.bundle.metadata["decision"].get("handoff", {}).get("validation_report", {}).get("report_id"),
    "own_real_optimization_executed": False,
    "synthetic_consistency": SYNTHETIC_CONSISTENCY, "real_metrics": None,
    "limitations": (["evaluación real 07 pendiente", "modelo definitivo no aprobado",
                     "prueba original ya consultada; no completamente independiente"]
        if SESSION is None or SESSION.controls.diagnostic_mode
        else SESSION.bundle.metadata["decision"].get("limitations", ["Prueba original previamente expuesta"]))}
print(json.dumps(REPORT_09, ensure_ascii=False, indent=2))


## 6 · Integración posterior con la aplicación

La interfaz podrá abrir una InferenceSession y llamar predict_clients: no debe reconstruir transformadores, usar el antiguo load_model sin verificación, descargar champion automáticamente ni reemplazar silenciosamente el pipeline. Mantener la validación y los controles del servicio, incluso cuando se utilice fuera del notebook.

Antes de uso real: finalizar 07, aprobar modelo y umbral, persistir en 08, revisar procedencia y dependencias y comprobar carga en una sesión nueva de Databricks. Indicar el directorio autorizado (por ejemplo Volume), su huella externa y permisos. No se realiza despliegue ni se afirma haber probado la ejecución integral remota de 06–09.
